# Chargement dans Azure SQL Database

Ce notebook fait suite à 03_schema_entrepot.ipynb.

L'objectif est de charger les 3 tables du star schema (DimDate, DimMeter,
FactConsumption) dans une base Azure SQL Database, ce qui constitue la couche
"entrepôt de données" de notre architecture Azure.

## Ressources Azure utilisées

Nous utilisons l'offre gratuite Azure SQL Database disponible pour toute
subscription Azure, qui inclut chaque mois :

- 100 000 vCore-secondes de calcul Serverless (≈ 28 heures à 1 vCore)
- 32 GB de stockage de données
- 32 GB de stockage de sauvegardes
- Jusqu'à 10 bases de données par subscription, sans limite de durée

Notre base energy-bi-db est configurée en General Purpose Serverless (Gen5, 2 vCores)
avec l'option "Auto-pause when free limits reached" activée, ce qui garantit
zéro frais tant qu'on reste dans les limites mensuelles.

## Prérequis complétés

- Base energy-bi-db créée sur le serveur energy-bi-server (Canada Central)
- IP locale ajoutée aux règles de firewall du serveur Azure SQL
- Driver ODBC Driver 18 for SQL Server installé sur le poste local
- Librairies Python installées :

bash
pip install pyodbc sqlalchemy

## Test de connexion

In [1]:

import os
from urllib.parse import quote_plus

# Paramètres de connexion Azure SQL
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

SERVER = os.getenv("AZURE_SQL_SERVER")
DATABASE = os.getenv("AZURE_SQL_DATABASE")
USERNAME = os.getenv("AZURE_SQL_USERNAME")
PASSWORD = os.getenv("AZURE_SQL_PASSWORD")
DRIVER = "ODBC Driver 18 for SQL Server"

# Chaîne de connexion SQLAlchemy
params = quote_plus(
    f"DRIVER={{{DRIVER}}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"UID={USERNAME};"
    f"PWD={PASSWORD};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=30;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# Test de connexion
with engine.connect() as conn:
    result = conn.execute(text("SELECT @@VERSION"))
    print(result.fetchone()[0])

Microsoft SQL Azure (RTM) - 12.0.2000.8 
	Apr 14 2026 20:27:12 
	Copyright (C) 2025 Microsoft Corporation



La connexion Python > Azure SQL Database est établie avec succès via SQLAlchemy + pyodbc.

La sortie affiche Microsoft SQL Azure (RTM) - 12.0.2000.8 : ce numéro de build est
affiché par tous les services Azure SQL Database indépendamment de la version réelle.
Azure SQL est un service toujours à jour géré par Microsoft, il n'y a pas de version
figée comme pour SQL Server on-premise.

Les paramètres de connexion sont chargés depuis un fichier .env via python-dotenv,
ce qui évite d'exposer les credentials dans le code versionné sur GitHub.m

## Création des tables dans Azure SQL Database

Nous créons les 3 tables du star schema dans Azure SQL via des instructions T-SQL (DDL).
Nous utilisons IF NOT EXISTS pour que le script soit rejouable sans erreur.

In [2]:
with engine.begin() as conn:
    # Supprimer dans l'ordre inverse des dépendances
    conn.execute(text("IF OBJECT_ID('FactConsumption','U') IS NOT NULL DROP TABLE FactConsumption"))
    conn.execute(text("IF OBJECT_ID('DimMeter','U') IS NOT NULL DROP TABLE DimMeter"))
    conn.execute(text("IF OBJECT_ID('DimDate','U') IS NOT NULL DROP TABLE DimDate"))

    conn.execute(text("""
                      CREATE TABLE DimDate
                      (
                          date_id          INT PRIMARY KEY,
                          consumption_date DATE     NOT NULL,
                          year             SMALLINT NOT NULL,
                          quarter          TINYINT  NOT NULL,
                          month            TINYINT  NOT NULL,
                          month_name       NVARCHAR(20) NOT NULL,
                          week             TINYINT  NOT NULL,
                          day_of_week      TINYINT  NOT NULL,
                          day_name         NVARCHAR(20) NOT NULL,
                          is_weekend       TINYINT  NOT NULL
                      )
                      """))

    conn.execute(text("""
                      CREATE TABLE DimMeter
                      (
                          meter_id   INT PRIMARY KEY,
                          meter_name NVARCHAR(50) NOT NULL,
                          location   NVARCHAR(100) NOT NULL,
                          equipment  NVARCHAR(255) NOT NULL,
                          unit       NVARCHAR(10) NOT NULL
                      )
                      """))

    conn.execute(text("""
                      CREATE TABLE FactConsumption
                      (
                          fact_id                    INT PRIMARY KEY IDENTITY(1,1),
                          date_id                    INT   NOT NULL REFERENCES DimDate (date_id),
                          meter_id                   INT   NOT NULL REFERENCES DimMeter (meter_id),
                          consumption_date           DATE  NOT NULL,
                          energy_wh                  FLOAT NOT NULL,
                          Global_active_power_mean   FLOAT,
                          Global_active_power_sum    FLOAT,
                          Global_reactive_power_mean FLOAT,
                          Voltage_mean               FLOAT,
                          Global_intensity_mean      FLOAT
                      )
                      """))

print("Tables supprimées et recréées avec succès.")

Tables supprimées et recréées avec succès.


Le code utilise T-SQL (Transact-SQL), le langage SQL de Microsoft utilisé par Azure SQL
Database.

DimDate:

Crée la table de dimension temporelle si elle n'existe pas déjà.
Elle contient une ligne par jour du dataset, enrichie d'attributs comme l'année, le trimestre,
le mois, la semaine et le type de jour (semaine/week-end). La colonne date_id (entier au
format YYYYMMDD) est la clé primaire qui servira de lien avec la table de faits.

DimMeter:

Crée la table de dimension compteur.
Elle contient 3 lignes fixes décrivant les sous-compteurs du foyer (cuisine, buanderie,
chauffe-eau/climatisation). meter_id est la clé primaire.

FactConsumption:

Crée la table de faits centrale du star schema.
fact_id est une clé primaire auto-incrémentée (IDENTITY(1,1) : Azure SQL génère
automatiquement une valeur unique à chaque insertion).
Les colonnes date_id et meter_id sont des clés étrangères (REFERENCES) qui
relient chaque fait à sa dimension date et à son compteur, garantissant l'intégrité
référentielle de l'entrepôt.

## Rechargement des tables du schéma

In [3]:
import pandas as pd
from pathlib import Path

project_root = Path().resolve().parent
schema_path = project_root / "data" / "schema"

# Rechargement des 3 tables exportées dans le notebook 03
dim_date = pd.read_csv(schema_path / "DimDate.csv")
dim_meter = pd.read_csv(schema_path / "DimMeter.csv")
fact_consumption = pd.read_csv(schema_path / "FactConsumption.csv")

print("DimDate :", dim_date.shape)
print("DimMeter :", dim_meter.shape)
print("FactConsumption :", fact_consumption.shape)

DimDate : (1442, 10)
DimMeter : (3, 5)
FactConsumption : (4326, 9)


## Chargement des données dans Azure SQL

Nous utilisons pandas.to_sql() pour insérer les 3 tables du star schema dans Azure SQL Database.

In [4]:
dim_date_to_load = dim_date.rename(columns={"date": "consumption_date"})
fact_to_load = fact_consumption.rename(columns={"date": "consumption_date"})

dim_date_to_load.to_sql("DimDate", con=engine, if_exists="append", index=False)
print(f"DimDate chargé : {len(dim_date_to_load)} lignes")

dim_meter.to_sql("DimMeter", con=engine, if_exists="append", index=False)
print(f"DimMeter chargé : {len(dim_meter)} lignes")

fact_to_load.to_sql("FactConsumption", con=engine, if_exists="append", index=False, chunksize=500)
print(f"FactConsumption chargé : {len(fact_to_load)} lignes")

DimDate chargé : 1442 lignes
DimMeter chargé : 3 lignes
FactConsumption chargé : 4326 lignes


## Vérification du chargement

In [5]:
with engine.connect() as conn:
    # Nombre de lignes par table
    for table in ["DimDate", "DimMeter", "FactConsumption"]:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        print(f"{table} : {count} lignes")

    print()

    # Vérification d'intégrité : jointure entre les 3 tables
    result = conn.execute(text("""
        SELECT TOP 5
            d.consumption_date, m.location,
            f.energy_wh,
            f.Global_active_power_mean
        FROM FactConsumption f
        JOIN DimDate d ON f.date_id = d.date_id
        JOIN DimMeter m ON f.meter_id = m.meter_id
        ORDER BY d.consumption_date, m.meter_id
        """))

    print("\nAperçu jointure :")
    for row in result:
        print(row)

DimDate : 1442 lignes
DimMeter : 3 lignes
FactConsumption : 4326 lignes


Aperçu jointure :
(datetime.date(2006, 12, 16), 'Cuisine', 0.0, 3.0534747474747475)
(datetime.date(2006, 12, 16), 'Buanderie', 546.0, 3.0534747474747475)
(datetime.date(2006, 12, 16), 'Chauffe-eau / Climatisation', 4926.0, 3.0534747474747475)
(datetime.date(2006, 12, 17), 'Cuisine', 2033.0, 2.354486111111111)
(datetime.date(2006, 12, 17), 'Buanderie', 4187.0, 2.354486111111111)


Le comptage confirme que les 3 tables sont bien chargées sans perte de données :
- DimDate : 1 442 lignes
- DimMeter : 3 lignes
- FactConsumption : 4 326 lignes

La jointure entre les 3 tables fonctionne correctement :
les clés étrangères date_id et meter_id relient bien chaque fait à sa date et
à son compteur. Les données affichées sont cohérentes avec ce qu'on attendait :
le 16 décembre 2006, la cuisine affiche 0 Wh (pas d'usage), la buanderie 546 Wh,
et le chauffe-eau/clim 4 926 Wh en plein hiver.

## Conclusion

Ce notebook a complété la couche entrepôt de données de notre architecture Azure.

Ce que nous avons accompli :

1. Connexion sécurisée à Azure SQL Database via SQLAlchemy + pyodbc,
avec les credentials chargés depuis un fichier .env (jamais dans le code).
2. Création du schéma DDL en T-SQL : 3 tables avec clés primaires, clés étrangères
et types de données adaptés (TINYINT, SMALLINT, NVARCHAR, IDENTITY).
3. Chargement des 3 tables via pandas.to_sql() avec if_exists="append".
4. Vérification par jointure SQL confirmant l'intégrité référentielle du star schema.

Problème rencontré et résolu :
date est un mot réservé T-SQL - la colonne a été renommée consumption_date
dans DimDate et FactConsumption avant l'insertion.